# Stroop (1935) color-word interference: analysis

Reads one run folder (`*_trials.csv`) and computes the original paper's measures: the time per
100-item sheet for each test, the interference D (interference test minus its control) and D
over the control, beside Stroop's own values (reprint Tables I and III). Trial-level measures
(median reaction time, error rate) follow.

Set `RUN` to a run folder; by default it uses the newest run under `data/` or `data-rehearsal/`.
A simulated run (`--mode simulate`) is software verification: its numbers come from the
simulated subject, whose medians were set from Stroop's means, so agreement with the paper
means nothing there.

In [ ]:
import os
from pathlib import Path

import matplotlib.pyplot as plt

from stroop1935.analysis import interference, item_measures, load_trials, sheet_times, test_items

HERE = Path.cwd() if (Path.cwd() / 'run.py').exists() else Path.cwd().parent
RUN = os.environ.get('ANALYSIS_RUN')
if RUN is None:
    runs = sorted(HERE.glob('data*/**/*_trials.csv'), key=lambda p: p.stat().st_mtime)
    RUN = runs[-1].parent if runs else None
print('run:', RUN)

In [ ]:
trials = load_trials(RUN)
print(trials['respond_to'].iloc[0], 'task; block order', trials['block_order'].iloc[0])
trials.groupby(['block', 'phase', 'test', 'outcome']).size().rename('trials').to_frame()

## Sheet times (Stroop's measure)

Each block is one sheet. `time_s` sums each item's time from onset to the right key; an item
whose error was never corrected is charged twice the sheet's mean time per item (Stroop's rule,
p. 17). `per_100_s` scales a sheet to 100 items (a rehearsal serves only a few). The sum leaves
out the time between items, so compare D over the control with the paper, not the seconds.

In [ ]:
sheets = sheet_times(trials)
sheets

In [ ]:
comparison = interference(sheets)
comparison.T

## Trial-level measures

In [ ]:
item_measures(trials)

In [ ]:
items = test_items(trials)
tests = list(dict.fromkeys(items['test']))
fig, axes = plt.subplots(1, 2, figsize=(10, 3.6))
for test in tests:
    right = items['first_correct'].fillna(False).astype(bool)
    axes[0].hist(items.loc[(items['test'] == test) & right, 'rt_ms'], bins=20, alpha=0.6, label=test)
axes[0].set_xlabel('reaction time of a right first key (ms)')
axes[0].set_ylabel('items')
axes[0].legend()
for block, g in items.groupby('block'):
    g = g.sort_values('position')
    axes[1].plot(g['position'], g['rt_correct_ms'].cumsum() / 1000.0,
                 label=f"block {block}: {g['test'].iloc[0]} form {g['form'].iloc[0]}")
axes[1].set_xlabel('item in sheet (reading order)')
axes[1].set_ylabel('cumulative time to right key (s)')
axes[1].legend(fontsize=8)
fig.tight_layout()

## Reading the result

- Colour naming (Experiment 2): Stroop found NCWd took 74% longer than NC (D/NC = 0.74).
- Word reading (Experiment 1): Stroop found RCNd only 5.6% slower than RCNb, not reliable.
  With keys instead of speech this comparison is a different task (docs/methods.md):
  colour interference on key responses to words is expected, so do not read a non-zero
  difference here as a failure to replicate Experiment 1, nor a zero one as a replication.
- One run is one subject. The paper's numbers are group means over 70 and 100 subjects.